In [ ]:
!pip install scikit-learn

In [ ]:
!pip install pyethnicity
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
import numpy as np
import pyethnicity
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from multiprocessing import Pool, cpu_count

#Preprocessing Dataset and Sampling

Keeping only the relevant columns from the dataset

In [ ]:
columns_to_keep = [
    'first_name',
    'last_name',
    'middle_name',
    'county_id',
    'county_desc',
    'race_code',
    'ethnic_code',
    'zip_code',
    'reason_cd',
    'party_cd',
    'gender_code',
    'age_at_year_end',
    'drivers_lic',
    'birth_state',
    'registr_dt'
]

print("Columns to keep defined:", columns_to_keep)

Columns to keep defined: ['first_name', 'last_name', 'middle_name', 'county_id', 'county_desc', 'race_code', 'ethnic_code', 'zip_code', 'reason_cd', 'party_cd', 'gender_code', 'age_at_year_end', 'drivers_lic', 'birth_state', 'registr_dt']


In [ ]:
base_path = "/content/drive/MyDrive/Fall 2025/Responsible AI/RAI Project/Project Implementation"
folder_path = base_path + "/ncvotersheets"
if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' does not exist. Please ensure it's created and accessible.")
else:
    all_files_and_dirs = os.listdir(folder_path)

    excel_files = [f for f in all_files_and_dirs if f.startswith('ncvoter') and f.endswith('.xlsx')]

Combining the 99 files into a single dataset.


In [ ]:
# List to store processed DataFrames
processed_dataframes = []

print("Starting to process Excel files...")
total = 99
count = 0
for file_name in excel_files:
    file_path = os.path.join(folder_path, file_name)
    try:

        df = pd.read_excel(file_path)
        count += 1

        existing_columns = [col for col in columns_to_keep if col in df.columns]
        processed_df = df[existing_columns]

        if 'reason_cd' in processed_df.columns:
            processed_df = processed_df[processed_df['reason_cd'] != 'RM']


        processed_dataframes.append(processed_df)

        print(f"Successfully processed '{file_name}'). Remaining: {total - count}")

    except FileNotFoundError:
        print(f"Error: File not found at '{file_path}'. Skipping this file.")
    except KeyError as e:
        print(f"Error processing '{file_name}': Missing expected column(s). {e}")
    except Exception as e:
        print(f"An unexpected error occurred while processing '{file_name}': {e}")

print(f"Finished processing {len(excel_files)} files. {len(processed_dataframes)} DataFrames were successfully processed.")


Starting to process Excel files...
Successfully processed 'ncvoter28.xlsx'). Remaining: 98
Successfully processed 'ncvoter1.xlsx'). Remaining: 97
Successfully processed 'ncvoter2.xlsx'). Remaining: 96
Successfully processed 'ncvoter10.xlsx'). Remaining: 95
Successfully processed 'ncvoter3.xlsx'). Remaining: 94
Successfully processed 'ncvoter29.xlsx'). Remaining: 93
Successfully processed 'ncvoter4.xlsx'). Remaining: 92
Successfully processed 'ncvoter30.xlsx'). Remaining: 91
Successfully processed 'ncvoter6.xlsx'). Remaining: 90
Successfully processed 'ncvoter31.xlsx'). Remaining: 89
Successfully processed 'ncvoter7.xlsx'). Remaining: 88
Successfully processed 'ncvoter8.xlsx'). Remaining: 87
Successfully processed 'ncvoter9.xlsx'). Remaining: 86
Successfully processed 'ncvoter37.xlsx'). Remaining: 85
Successfully processed 'ncvoter38.xlsx'). Remaining: 84
Successfully processed 'ncvoter40.xlsx'). Remaining: 83
Successfully processed 'ncvoter11.xlsx'). Remaining: 82
Successfully processe

In [ ]:
if processed_dataframes:
    combined_df = pd.concat(processed_dataframes, ignore_index=True)
    print(f"Combined all processed data into a single DataFrame with {len(combined_df)} rows.")
    print(combined_df.head())

Combined all processed data into a single DataFrame with 8546645 rows.
  first_name last_name middle_name  county_id county_desc race_code  \
0      AARON     AARON       JAMES         28        DARE         W   
1      SHANE     AARON      LANDON         28        DARE         W   
2       TARA     AARON     STAPLES         28        DARE         W   
3    ANTHONY     ABATE      JOSEPH         28        DARE         W   
4   LORRAINE     ABATE       MARIE         28        DARE         W   

  ethnic_code  zip_code reason_cd party_cd gender_code  age_at_year_end  \
0          UN   27943.0        AV      REP           M             55.0   
1          NL   27943.0        AV      UNA           M             29.0   
2          UN   27943.0        AV      UNA           F             58.0   
3          UN   27959.0        AV      UNA           M             67.0   
4          UN   27959.0        AV      REP           F             71.0   

  drivers_lic birth_state  registr_dt  
0          

Dropping columns with no zip code.

In [ ]:
initial_rows = len(combined_df)
combined_df = combined_df.dropna(subset=['zip_code'])
final_rows = len(combined_df)

print(f"Removed {initial_rows - final_rows} rows with empty or NaN zip codes.")
print(f"Combined DataFrame now has {final_rows} rows.")
print(combined_df.head())

Removed 944675 rows with empty or NaN zip codes.
Combined DataFrame now has 7601970 rows.
  first_name last_name middle_name  county_id county_desc race_code  \
0      AARON     AARON       JAMES         28        DARE         W   
1      SHANE     AARON      LANDON         28        DARE         W   
2       TARA     AARON     STAPLES         28        DARE         W   
3    ANTHONY     ABATE      JOSEPH         28        DARE         W   
4   LORRAINE     ABATE       MARIE         28        DARE         W   

  ethnic_code  zip_code reason_cd party_cd gender_code  age_at_year_end  \
0          UN   27943.0        AV      REP           M             55.0   
1          NL   27943.0        AV      UNA           M             29.0   
2          UN   27943.0        AV      UNA           F             58.0   
3          UN   27959.0        AV      UNA           M             67.0   
4          UN   27959.0        AV      REP           F             71.0   

  drivers_lic birth_state  regis

If the ethnic_code column contains HL and the race_code column contains (undesignated) then we will set the value of race_code column as HL.

In [ ]:
initial_u_hl_count = len(combined_df[(combined_df['ethnic_code'] == 'HL') & (combined_df['race_code'] == 'U')])

combined_df.loc[
    (combined_df['ethnic_code'] == 'HL') & (combined_df['race_code'] == 'U'),
    'race_code'
] = 'HL'

final_u_hl_count = len(combined_df[(combined_df['ethnic_code'] == 'HL') & (combined_df['race_code'] == 'U')])

print(f"Number of rows where ethnic_code was 'HL' and race_code was 'U' before update: {initial_u_hl_count}")
print(f"Number of rows updated: {initial_u_hl_count - final_u_hl_count}")
print(f"Number of rows where ethnic_code is 'HL' and race_code is still 'U' after update: {final_u_hl_count}")

# Display the first few rows to show the change
print("\nDataFrame head after update (showing relevant columns):")
print(combined_df[combined_df['ethnic_code'] == 'HL'][['ethnic_code', 'race_code']].head())

Number of rows where ethnic_code was 'HL' and race_code was 'U' before update: 58552
Number of rows updated: 58552
Number of rows where ethnic_code is 'HL' and race_code is still 'U' after update: 0

DataFrame head after update (showing relevant columns):
    ethnic_code race_code
30           HL        HL
35           HL         O
50           HL         W
61           HL         W
164          HL         W


In [ ]:
length = len(combined_df)
percent = 0.44
print("No. of row: " + str(length) + " Sample size: " + str(percent * length))
combined_df["strata"] = combined_df["party_cd"].astype(str) + "_" + combined_df["race_code"].astype(str)
counts = combined_df["strata"].value_counts()
rare_strata = counts[counts < 2].index
print("Rare strata:", rare_strata)
df_clean = combined_df[~combined_df["strata"].isin(rare_strata)]

df_sampled, _ = train_test_split(
    df_clean,
    train_size=percent,
    stratify=df_clean["strata"],
    random_state=42
)
print(len(df_sampled))

print(f"\nOriginal size: {len(combined_df)}")
print(f"Sampled size: {len(df_sampled)}")

print(combined_df['party_cd'].value_counts(normalize=True))
print(df_sampled['party_cd'].value_counts(normalize=True))

print(combined_df['race_code'].value_counts(normalize=True))
print(df_sampled['race_code'].value_counts(normalize=True))


sample_file_name = 'stratified_sample_' + str(len(df_sampled)) + '.csv'

df_sampled.to_csv(sample_file_name, index=False)

print(f"Successfully saved the new dataset to {sample_file_name}")

No. of row: 228059 Sample size: 100345.96
Rare strata: Index(['GRE_HL', 'GRE_I'], dtype='object', name='strata')
100345

Original size: 228059
Sampled size: 100345
party_cd
UNA    0.388450
DEM    0.306741
REP    0.298131
LIB    0.006122
GRE    0.000557
Name: proportion, dtype: float64
party_cd
UNA    0.388454
DEM    0.306755
REP    0.298148
LIB    0.006098
GRE    0.000545
Name: proportion, dtype: float64
race_code
W     0.631540
B     0.200704
U     0.089450
O     0.043105
A     0.018247
HL    0.007950
M     0.004887
I     0.003982
P     0.000136
Name: proportion, dtype: float64
race_code
W     0.631546
B     0.200705
U     0.089453
O     0.043107
A     0.018242
HL    0.007939
M     0.004884
I     0.003980
P     0.000144
Name: proportion, dtype: float64
Successfully saved the new dataset to stratified_sample_100345.csv


# Distribution of the dataset

Finding race distribution for the dataset.

In [ ]:
race_code_counts = df_sampled['race_code'].value_counts()
race_code_percentages = df_sampled['race_code'].value_counts(normalize=True) * 100

race_code_distribution = pd.DataFrame({
    'Count': race_code_counts,
    'Percentage': race_code_percentages
})

print("Race Code Distribution:")
print(race_code_distribution.round(2))


Race Code Distribution:
            Count  Percentage
race_code                    
W          465268       63.15
B          147863       20.07
U           65898        8.94
O           31758        4.31
A           13443        1.82
HL           5855        0.79
M            3599        0.49
I            2935        0.40
P             103        0.01


In [ ]:
display(race_code_distribution)

,Count,Percentage
race_code,,
W,465268,63.153808
B,147863,20.070393
U,65898,8.944758
O,31758,4.310717
A,13443,1.824705
HL,5855,0.794737
M,3599,0.488515
I,2935,0.398386
P,103,0.013981


Finding party distribution for dataset.

In [ ]:
party_code_counts = df_sampled['party_cd'].value_counts()
party_code_percentages = df_sampled['party_cd'].value_counts(normalize=True) * 100

party_code_distribution = pd.DataFrame({
    'Count': party_code_counts,
    'Percentage': party_code_percentages
})

print("Party Code Distribution:")
print(party_code_distribution.round(2))

Party Code Distribution:
           Count  Percentage
party_cd                    
UNA       286179       38.84
DEM       225986       30.67
REP       219642       29.81
LIB         4511        0.61
GRE          404        0.05


In [ ]:
display(party_code_distribution)

,Count,Percentage
party_cd,,
UNA,286179,38.844910
DEM,225986,30.674529
REP,219642,29.813417
LIB,4511,0.612307
GRE,404,0.054838


# Doing BISG on the sampled records

### Using the crosswalk file to map ZIP code to ZCTA.



In [ ]:
base_path = "/content/drive/MyDrive/Fall 2025/Responsible AI/RAI Project/Project Implementation"
crosswalk_file_path = os.path.join(base_path, 'ZIP Code to ZCTA Crosswalk.xlsx')
print(crosswalk_file_path)

/content/drive/MyDrive/Fall 2025/Responsible AI/RAI Project/Project Implementation/ZIP Code to ZCTA Crosswalk.xlsx


In [ ]:
try:
  crosswalk_df = pd.read_excel(crosswalk_file_path)
  print(f"Successfully loaded '{crosswalk_file_path}'.")
  print("Crosswalk DataFrame head:")
  print(crosswalk_df.head())

  df_sampled['zip_code'] = pd.to_numeric(df_sampled['zip_code'], errors='coerce')
  crosswalk_df['ZIP_CODE'] = pd.to_numeric(crosswalk_df['ZIP_CODE'], errors='coerce')

  df_with_zcta = pd.merge(
      df_sampled,
      crosswalk_df[['ZIP_CODE', 'zcta']],
      left_on='zip_code',
      right_on='ZIP_CODE',
      how='left'
  )

  df_with_zcta = df_with_zcta.drop(columns=['ZIP_CODE'], errors='ignore')

  df_with_zcta['zcta'] = df_with_zcta['zcta'].astype('Int64')
  df_with_zcta['zip_code'] = df_with_zcta['zip_code'].astype('Int64')

  print("\n'zcta' column added to df_with_zcta.")
  print(df_with_zcta.head())

except FileNotFoundError:
    print(f"Error: Crosswalk file not found at '{crosswalk_file_path}'. Please ensure the file exists.")
except Exception as e:
    print(f"An error occurred while processing the crosswalk file or merging: {e}")


Successfully loaded '/content/drive/MyDrive/Fall 2025/Responsible AI/RAI Project/Project Implementation/ZIP Code to ZCTA Crosswalk.xlsx'.
Crosswalk DataFrame head:
   ZIP_CODE        PO_NAME STATE       ZIP_TYPE     zcta     zip_join_type
0     77982  Port O Connor    TX  Zip Code Area  77982.0  Zip matches ZCTA
1     77983       Seadrift    TX  Zip Code Area  77983.0  Zip matches ZCTA
2     78860       El Indio    TX  Zip Code Area  78860.0  Zip matches ZCTA
3     77950       Austwell    TX  Zip Code Area  77950.0  Zip matches ZCTA
4     77990         Tivoli    TX  Zip Code Area  77990.0  Zip matches ZCTA

'zcta' column added to df_with_zcta.
  first_name last_name middle_name  county_id  county_desc race_code  \
0      MAKAJ   MCCURDY     ARIONNA         34      FORSYTH         U   
1    MELISSA      DYER       LEIGH         36       GASTON         W   
2     DENISE     SHARP     DELORES         21       CHOWAN         B   
3     DAVION     FALLS     MARQUIS         60  MECKLENBURG  

Drop any rows where the zcta value is NaN


In [ ]:
initial_rows = len(df_with_zcta)
df_with_zcta = df_with_zcta.dropna(subset=['zcta'])
final_rows = len(df_with_zcta)

print(f"Removed {initial_rows - final_rows} rows with empty or NaN 'zcta' values.")
print(f"DataFrame now has {final_rows} rows.")
print(df_with_zcta.head())

Removed 0 rows with empty or NaN 'zcta' values.
DataFrame now has 26245 rows.
  first_name last_name middle_name  county_id  county_desc race_code  \
0      MAKAJ   MCCURDY     ARIONNA         34      FORSYTH         U   
1    MELISSA      DYER       LEIGH         36       GASTON         W   
2     DENISE     SHARP     DELORES         21       CHOWAN         B   
3     DAVION     FALLS     MARQUIS         60  MECKLENBURG         B   
4    TIMOTHY   KILGORE       JAMES         10    BRUNSWICK         W   

  ethnic_code  zip_code reason_cd party_cd gender_code  age_at_year_end  \
0          UN     27110        DU      UNA           U             24.0   
1          NL     28012        AV      UNA           F             50.0   
2          UN     27932        AV      DEM           F             69.0   
3          UN     28216        AV      DEM           M             18.0   
4          NL     28465        AV      UNA           M             73.0   

  drivers_lic birth_state  registr_dt 

In [ ]:
df_with_zcta.to_csv('df_with_zcta.csv', index=False)
print("DataFrame 'df_with_zcta' saved to 'df_with_zcta.csv'")

DataFrame 'df_with_zcta' saved to 'df_with_zcta.csv'


BIFSG using Pyethnicity

In [ ]:
def bifsg(args):
    first_name, last_name, zcta_value, row_index = args

    result = pyethnicity.bifsg(
        first_name,
        last_name,
        zcta_value,
        "zcta"
    )

    return {
        "index": row_index,
        "prob_white": result["white"].item(),
        "prob_black": result["black"].item(),
        "prob_hisp": result["hispanic"].item(),
        "prob_asian": result["asian"].item()
    }

In [ ]:
tasks = []
for index, row in df_with_zcta.iterrows():
    tasks.append((str(row["last_name"]), row["zcta"], index))

In [ ]:
workers = cpu_count()-1

with Pool(workers) as pool:
    results = list(
        tqdm(
            pool.imap_unordered(bifsg, tasks),
            total=len(tasks),
            desc="Processing bifsg"
        )
    )


Processing bifsg: 100%|██████████| 26245/26245 [59:06<00:00,  7.40it/s]


In [ ]:
for res in results:
    idx = res["index"]
    df_with_zcta.loc[idx, "prob_white"] = res["prob_white"]
    df_with_zcta.loc[idx, "prob_black"] = res["prob_black"]
    df_with_zcta.loc[idx, "prob_hisp"] = res["prob_hisp"]
    df_with_zcta.loc[idx, "prob_asian"] = res["prob_asian"]

print(df_with_zcta.head() )
length = len(df_with_zcta)
df_with_zcta.to_csv('real_bisg_done_sample_' + str(length) + '.csv', index=False)

  first_name last_name middle_name  county_id  county_desc race_code  \
0      MAKAJ   MCCURDY     ARIONNA         34      FORSYTH         U   
1    MELISSA      DYER       LEIGH         36       GASTON         W   
2     DENISE     SHARP     DELORES         21       CHOWAN         B   
3     DAVION     FALLS     MARQUIS         60  MECKLENBURG         B   
4    TIMOTHY   KILGORE       JAMES         10    BRUNSWICK         W   

  ethnic_code  zip_code reason_cd party_cd  ... age_at_year_end  drivers_lic  \
0          UN     27110        DU      UNA  ...            24.0            N   
1          NL     28012        AV      UNA  ...            50.0            Y   
2          UN     27932        AV      DEM  ...            69.0            Y   
3          UN     28216        AV      DEM  ...            18.0            Y   
4          NL     28465        AV      UNA  ...            73.0            Y   

  birth_state  registr_dt strata   zcta  prob_white prob_black prob_hisp  \
0         